# 02. Матрицы расстояний: по прямой и по дорогам

Расстояние по прямой считается мгновенно и не требует инфраструктуры, но
систематически занижает пробег. Дорожное расстояние из OSRM — то, что
реально проедет машина.

Ноутбук работает и без OSRM: раздел с дорожными расстояниями просто
пропускается. Как поднять сервер — см. [docs/osrm.md](../docs/osrm.md).

In [ ]:
import asyncio
import sys
import time

sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from routeforge.distance import haversine_distance, haversine_matrix, osrm_matrix
from routeforge.io import read_points

sites = read_points("../data/sample/sites.csv")
coords = list(sites[["lat", "lon"]].itertuples(index=False, name=None))
print(len(coords), "точек")

## Haversine

Считается векторно, поэтому размер матрицы почти не мешает.

In [ ]:
t = time.perf_counter()
matrix = haversine_matrix(coords, coords)
print(f"{matrix.shape} за {time.perf_counter() - t:.3f} с")

# Диагональ нулевая, матрица симметрична.
assert (np.diag(matrix) == 0).all()
assert np.allclose(matrix, matrix.T)
print("расстояние между первыми двумя точками: %.2f км" % (matrix[0, 1] / 1000))

In [ ]:
# Скалярная функция согласована с матричной.
print("%.4f == %.4f" % (haversine_distance(coords[0], coords[1]), matrix[0, 1]))

## Сколько времени занимает матрица нарастающего размера

In [ ]:
for n in (100, 250, 500, 1000, 2000):
    pts = coords * (n // len(coords) + 1)
    pts = pts[:n]
    t = time.perf_counter()
    haversine_matrix(pts, pts)
    print(f"{n:5} x {n:<5} -> {time.perf_counter() - t:6.3f} с")

## OSRM

Матрица берётся через `/table`: один запрос возвращает целую строку, а не
одну ячейку. На задаче «3 базы x 240 точек» это 9 запросов вместо 720.

Если сервер не поднят, ячейка ниже честно об этом скажет и пойдёт дальше.

In [ ]:
OSRM_URL = "http://localhost:5000"

depots_df = pd.read_csv("../data/sample/depots.csv")
depots = list(depots_df[["lat", "lon"]].itertuples(index=False, name=None))

road = None
try:
    t = time.perf_counter()
    road = asyncio.run(osrm_matrix(depots, coords, OSRM_URL))
    print(f"{road.shape} за {time.perf_counter() - t:.2f} с")
except Exception as exc:
    print(f"OSRM недоступен ({type(exc).__name__}), раздел пропущен.")
    print("Поднять сервер: см. docs/osrm.md")

## Насколько дорога длиннее прямой

Отношение дорожного расстояния к прямому — коэффициент извилистости.
На городской сети он обычно 1.2–1.5; значение, сильно выбивающееся вверх,
означает препятствие: реку, железную дорогу, промзону.

In [ ]:
if road is not None:
    straight = haversine_matrix(depots, coords)
    ratio = np.where(straight > 0, road / np.maximum(straight, 1), np.nan)
    finite = ratio[np.isfinite(ratio)]
    print("коэффициент извилистости: медиана %.2f, 90-й процентиль %.2f, максимум %.2f"
          % (np.median(finite), np.percentile(finite, 90), finite.max()))

    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 4))
    plt.hist(finite, bins=40, color="steelblue")
    plt.axvline(np.median(finite), c="k", ls="--", label="медиана")
    plt.xlabel("дорожное / прямое")
    plt.ylabel("пар точек")
    plt.legend()
    plt.grid(alpha=0.2)
else:
    print("нужен OSRM")

## Что из этого следует

Для **распределения по базам** haversine достаточно: важен порядок близости,
а он почти всегда совпадает с дорожным.

Для **построения маршрутов** разница уже значима: маршрут, оптимальный по
прямой, на дорогах может оказаться заметно хуже — особенно там, где
извилистость распределена неравномерно (река, объезд промзоны).